In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
catalog = "bank_catalog"

bronze = f"{catalog}.bronze"
silver = f"{catalog}.silver"
gold = f"{catalog}.gold"
quarantine = f"{catalog}.quarantine"

quality_table = f"{gold}.gold_data_quality_report"

datasets = [
    "customers",
    "accounts",
    "transactions",
    "loans",
    "branches"
]

print("Data Quality configuration completed")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {quality_table} (
    dataset STRING,
    batch_id STRING,
    bronze_count BIGINT,
    silver_count BIGINT,
    rejected_count BIGINT,
    null_count BIGINT,
    duplicate_count BIGINT,
    quality_status STRING,
    check_timestamp TIMESTAMP
)
USING DELTA
""")

In [0]:
def get_count(table_name, batch_id=None):

    if not spark.catalog.tableExists(table_name):
        return 0

    df = spark.table(table_name)

    if batch_id is not None:
        df = df.filter(F.col("batch_id") == batch_id)

    return df.count()

In [0]:
def count_nulls(df):

    expressions = [
        F.sum(
            F.when(
                F.col(c).isNull(),
                1
            ).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]

    result = df.agg(*expressions).first()

    return sum(
        value or 0 for value in result
    )

In [0]:
primary_keys = {
    "customers": "customer_id",
    "accounts": "account_id",
    "transactions": "transaction_id",
    "loans": "loan_id",
    "branches": "branch_id"
}


def count_duplicates(df, primary_key):

    duplicate_count = (
        df.groupBy(primary_key)
        .count()
        .filter(F.col("count") > 1)
        .agg(
            F.sum(F.col("count") - 1).alias("duplicates")
        )
        .first()["duplicates"]
    )

    return duplicate_count or 0

In [0]:
quality_results = []

for dataset in datasets:

    bronze_table = f"{bronze}.{dataset}"
    silver_table = f"{silver}.{dataset}"
    quarantine_table = f"{quarantine}.{dataset}_rejected"

    bronze_df = spark.table(bronze_table)
    silver_df = spark.table(silver_table)

    batches = [
        row["batch_id"]
        for row in bronze_df
        .select("batch_id")
        .distinct()
        .collect()
    ]

    primary_key = primary_keys[dataset]

    for batch_id in batches:

        bronze_batch = bronze_df.filter(
            F.col("batch_id") == batch_id
        )

        silver_batch = silver_df.filter(
            F.col("batch_id") == batch_id
        )

        bronze_count = bronze_batch.count()
        silver_count = silver_batch.count()

        rejected_count = 0

        if spark.catalog.tableExists(quarantine_table):

            rejected_count = (
                spark.table(quarantine_table)
                .filter(F.col("batch_id") == batch_id)
                .count()
            )

        null_count = count_nulls(silver_batch)

        duplicate_count = count_duplicates(
            silver_batch,
            primary_key
        )

        status = (
            "PASS"
            if duplicate_count == 0
            else "FAIL"
        )

        quality_results.append((
            dataset,
            batch_id,
            bronze_count,
            silver_count,
            rejected_count,
            null_count,
            duplicate_count,
            status,
            datetime.now()
        ))

print("Quality checks completed")

In [0]:
quality_schema = """
dataset STRING,
batch_id STRING,
bronze_count LONG,
silver_count LONG,
rejected_count LONG,
null_count LONG,
duplicate_count LONG,
quality_status STRING,
check_timestamp TIMESTAMP
"""

quality_df = spark.createDataFrame(
    quality_results,
    schema=quality_schema
)

quality_df.write.format("delta").mode("append").saveAsTable(
    quality_table
)

display(quality_df)

In [0]:
gold_tables = {
    "gold_customer_360": "customer_id",
    "gold_account_summary": "account_id",
    "gold_transaction_details": "transaction_id",
    "gold_loan_analytics": "loan_id",
    "gold_branch_performance": "branch_id",
    "gold_banking_kpis": "kpi_id"
}

for table, key in gold_tables.items():

    table_name = f"Verse_commonwealth_bank_catalog.gold.{table}"

    df = spark.table(table_name)

    print(f"\nChecking {table}")

    print("Total records:", df.count())

    print(
        "Duplicate records:",
        count_duplicates(df, key)
    )

    print("Null primary keys:",
          df.filter(F.col(key).isNull()).count())

In [0]:
%sql
SELECT
    dataset,
    batch_id,
    bronze_count,
    silver_count,
    rejected_count,
    null_count,
    duplicate_count,
    quality_status,
    check_timestamp
FROM bank_catalog.gold.gold_data_quality_report
ORDER BY check_timestamp DESC, dataset, batch_id;